<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/BaselineByteTrack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline · YOLO + ByteTrack tracking (not part of the pipeline)

The **first version of the tracking step**, kept only to compare with SAM 2.1 (step 03). YOLO26n boxes the cow at 5 fps and **ByteTrack** links boxes by position; it gives a new track whenever YOLO loses her for > 10 s, so one cow became **47 tracks**. Takes ~20 min on a T4. You only need it again for a new video you want to compare.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/BaselineByteTrack/`, **overwriting** each run:

- `Tracks.csv`: one row per box per processed frame (step 03 compares with it)
- `TrackSummary.csv`: one row per track
- `PreviewTracks.mp4`: boxes and TrackIds drawn, 4x speed

Needs a **T4 GPU**. The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install YOLO.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Baselines.ByteTrack import DetectAndTrack
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
ClipPath = VideoOutputDir / "ReferenceClip.mp4"   # step 01
OutputDir = VideoOutputDir / "BaselineByteTrack"   # earlier runs: rename the old Step02bFull folder to this

TargetFps = 5          # frames per second processed
Confidence = 0.25      # lower = more boxes (and more false ones)
MaxSeconds = None      # None = the whole clip
PreviewSpeed = 4

assert ClipPath.exists(), f"Run step 01 first: {ClipPath}"

In [ ]:
# 3. Run: detect + track, then show the summary and the preview.
import pandas as pd
from IPython.display import Video, display

Stats = DetectAndTrack(ClipPath, OutputDir, TargetFps=TargetFps, Confidence=Confidence, MaxSeconds=MaxSeconds,
                       PreviewSpeed=PreviewSpeed)

display(pd.read_csv(OutputDir / "TrackSummary.csv"))
Path = OutputDir / "PreviewTracks.mp4"
if Path.exists():
    Mb = Path.stat().st_size / 2**20
    print(f"Preview: {Path} ({Mb:.0f} MB)")
    if Mb < 50:   # bigger files are too heavy to play inside the notebook; open them from Drive
        display(Video(str(Path), embed=True, width=960))

## About this step

**Models:** **YOLO26n** (COCO "cow" class) + **ByteTrack** (links boxes across frames by position), 1 of every 6 frames (5 fps).

**What we found (whole reference clip, 1 h):** one cow, **47 tracks**. Long tracks while she stands or lies calmly (e.g. T5 51–401 s, T198 2516–3256 s); many short, unsure tracks between ~630 and 2500 s while she lies and strains. 18,603 frames in ~20 min on a T4. This is why the pipeline tracks with SAM 2.1 instead (step 03).

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.